**Polycystic Ovary Syndrome (PCOS) Prediction
bold text**

In [ ]:
from google.colab import userdata
github_token = userdata.get("GITHUB_TOKEN")
!git config --global user.name "roshinsthomas"
!git config --global user.email "roshinsthomas@gmail.com"
# !git remote set-url origin https://{github_token}@github.com/roshinsthomas/pcos_prediction.git

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
#import data
pcos_df = pd.read_csv('/content/drive/MyDrive/datasets/PCOS.csv')
pcos_df.head(3)
pcos_df.info()
print("Dataset shape :",pcos_df.shape)
pcos_df.isna().sum().sum()

In [ ]:
# Performing EDA
pcos_df = pcos_df.drop(columns = ['Sl. No','Patient File No.','Unnamed: 44'])
pcos_df.head(2)


In [ ]:
nan_rows = pcos_df[pcos_df.isna().any(axis=1)]
print(nan_rows)

In [ ]:
pcos_df['Marraige Status (Yrs)']= pcos_df['Marraige Status (Yrs)'].fillna(pcos_df['Marraige Status (Yrs)'].median())
pcos_df['Fast food (Y/N)']= pcos_df['Fast food (Y/N)'].fillna(pcos_df['Fast food (Y/N)'].median())
pcos_df.isna().sum()

In [ ]:
pcos_df.select_dtypes(include= ['object']).head()

In [ ]:
pcos_df['II    beta-HCG(mIU/mL)'] = pd.to_numeric(pcos_df['II    beta-HCG(mIU/mL)'], errors = 'coerce')
pcos_df['AMH(ng/mL)'] = pd.to_numeric(pcos_df['AMH(ng/mL)'], errors = 'coerce')

pcos_df.columns = pcos_df.columns.str.strip()

In [ ]:
pcos_df['II    beta-HCG(mIU/mL)'] = pcos_df['II    beta-HCG(mIU/mL)'].fillna(pcos_df['II    beta-HCG(mIU/mL)'].median())
pcos_df['AMH(ng/mL)'] = pcos_df['AMH(ng/mL)'].fillna(pcos_df['AMH(ng/mL)'].median())
pcos_df.isna().sum()[pcos_df.isna().sum()>0]

In [ ]:
pcos_df.info()

In [ ]:
print(pcos_df.duplicated().sum())
pcos_df.drop_duplicates()

In [ ]:
# Detect & Handle Outliers
# Visualize numerical distributions to spot extreme outliers
plt.figure(figsize=(25, 10))
sns.boxplot(data=pcos_df)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Apply Winsorization to specific columns with extreme values
cols_to_winsorize = ['II    beta-HCG(mIU/mL)', 'No. of aborptions']

#capping extreme high values at the 99th percentile ceiling.
for col in cols_to_winsorize:
    upper_limit = pcos_df[col].quantile(0.99)
    pcos_df[col] = np.where(pcos_df[col] > upper_limit, upper_limit, pcos_df[col])

In [ ]:
#splitting and scaling data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

from sklearn.model_selection import train_test_split

x = pcos_df.drop(columns = ['PCOS (Y/N)'])
y = pcos_df['PCOS (Y/N)']

x_train, x_test, y_train, y_test = train_test_split(
    x,y, test_size = 0.2, random_state= 42)

x_train_scaled = pd.DataFrame(scaler.fit_transform(x_train), columns = x.columns)
x_test_scaled = pd.DataFrame(scaler.transform(x_test), columns = x.columns)

In [ ]:
#since the class is a little imbalance, we use smote
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state = 42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train_scaled, y_train)

In [ ]:
plt.figure(figsize=(10,7))

plt.subplot(1,2,1)
sns.countplot(x=y)
plt.title("Class Distribution before SMOTE")
plt.grid(True)


plt.subplot(1,2,2)
sns.countplot(x=y_train_resampled)
plt.title("Class Distribution after SMOTE")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

input_shape = x_train_resampled.shape[1]
model = keras.Sequential([
    #Input layer
    layers.Input(shape = (input_shape,)),

    #first hidden layer
    layers.Dense(64, activation = 'relu'),
    layers.Dropout(0.2),

    #second hidden layer
    layers.Dense(32, activation = 'relu'),
    layers.Dropout(0.2),

    # Output layer for Binary Classifications
    # using 1 neuron with sigmoid activation to get output probability between 0 and 1
    layers.Dense(1, activation = 'sigmoid')
])

model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy',
               tf.keras.metrics.Recall(name = 'recall'),
               tf.keras.metrics.Precision(name = 'precision')]
)

In [ ]:
#Training the model
history = model.fit(
    x_train_resampled,
    y_train_resampled,
    epochs = 50,
    batch_size = 32,
    validation_split = 0.2,
    verbose = 1
)

In [ ]:
#Get probabilities
y_pred_probabilities = model.predict(x_test_scaled)
#converting the probabilities to binary class labels
y_pred = (y_pred_probabilities > 0.5).astype(int)

In [ ]:
#generating metrics
from sklearn.metrics import classification_report, confusion_matrix
print("Classification Report\n",classification_report(y_test, y_pred))
print("Confusion matrix\n",confusion_matrix(y_test, y_pred))


In [ ]:
plt.figure(figsize = (14,5))

plt.subplot(1,2,1)
plt.plot(history.history['loss'], label = 'Training loss')
plt.plot(history.history['val_loss'], label = "Validation Loss")

plt.title ("Training and Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label = "Training Accuracy")
plt.plot(history.history['val_accuracy'], label = "Validation Accuracy")

plt.title('Training vs Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [29]:
%cd /content/drive/MyDrive/Projects/pcos_prediction/

/content/drive/MyDrive/Projects/pcos_prediction


In [ ]:
!ls